In [94]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

%matplotlib inline

In [95]:
df = pd.read_csv('vacancies.csv')

In [96]:
df.head()

,vacancy_id,salary_from,salary_to,experience,city,employment_type
0,37351176,60000.0,70000.0,Нет опыта,Екатеринбург,Удалённая работа
1,53503796,140000.0,210000.0,От 1 года до 3 лет,Москва,Полная занятость
2,20319022,NaN,NaN,Более 6 лет,Москва,Полная занятость
3,40619665,145000.0,215000.0,От 1 года до 3 лет,Нижний Новгород,Полная занятость
4,14675717,NaN,NaN,Более 6 лет,Москва,Полная занятость


In [97]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   vacancy_id       100000 non-null  int64  
 1   salary_from      68385 non-null   float64
 2   salary_to        69219 non-null   float64
 3   experience       100000 non-null  object 
 4   city             100000 non-null  object 
 5   employment_type  100000 non-null  object 
dtypes: float64(2), int64(1), object(3)
memory usage: 4.6+ MB


In [98]:
df.describe()

,vacancy_id,salary_from,salary_to
count,1.000000e+05,6.838500e+04,6.921900e+04
mean,5.502163e+07,4.547105e+05,6.155720e+05
std,2.595831e+07,2.466358e+06,3.377744e+06
min,1.000028e+07,5.000000e+03,7.000000e+03
25%,3.250663e+07,8.500000e+04,1.100000e+05
50%,5.510610e+07,1.200000e+05,1.700000e+05
75%,7.749584e+07,1.900000e+05,2.600000e+05
max,9.999938e+07,5.950000e+07,8.950000e+07


Проверим дубликаты в vacanty_id

In [99]:
df['vacancy_id'].nunique()

100000

Проверим на корректность заполнения данных

In [100]:
mask = df['salary_from'] > df['salary_to']
df[mask]

,vacancy_id,salary_from,salary_to,experience,city,employment_type
78,48429297,165000.0,130000.0,От 3 до 6 лет,Екатеринбург,Полная занятость
129,26376291,50000.0,45000.0,Нет опыта,Екатеринбург,Стажировка
200,51907364,355000.0,230000.0,От 1 года до 3 лет,Москва,Полная занятость
248,96998676,220000.0,160000.0,Более 6 лет,Новосибирск,Полная занятость
293,25983334,195000.0,150000.0,От 1 года до 3 лет,Москва,Полная занятость
...,...,...,...,...,...,...
99307,88048751,110000.0,80000.0,От 1 года до 3 лет,Казань,Полная занятость
99469,26281449,65000.0,60000.0,Нет опыта,Нижний Новгород,Полная занятость
99982,12506813,320000.0,200000.0,От 1 года до 3 лет,Москва,Полная занятость
99991,55437215,45000.0,40000.0,Нет опыта,Казань,Стажировка


Поменяем местами salary_from и salary_to, где нарушен порядок

In [101]:
df.loc[mask, ['salary_from', 'salary_to']] = (
    df.loc[mask, ['salary_to', 'salary_from']].to_numpy()
)

In [102]:
df[df['salary_from'] > df['salary_to']]

,vacancy_id,salary_from,salary_to,experience,city,employment_type


Проверим, есть ли отрицательные или "нулевые" зарплаты 

In [103]:
df[df['salary_from'] <= 0]

,vacancy_id,salary_from,salary_to,experience,city,employment_type


In [104]:
df[df['salary_to'] <= 0]

,vacancy_id,salary_from,salary_to,experience,city,employment_type


Проверим, нет ли в "строковых" столбцах значений, которые "неправильно" написаны

In [105]:
df['experience'].unique()

array(['Нет опыта', 'От 1 года до 3 лет', 'Более 6 лет', 'От 3 до 6 лет'],
      dtype=object)

In [106]:
df['city'].unique()

array(['Екатеринбург', 'Москва', 'Нижний Новгород', 'Санкт-Петербург',
       'Новосибирск', 'Казань'], dtype=object)

In [107]:
df['employment_type'].unique()

array(['Удалённая работа', 'Полная занятость', 'Стажировка',
       'Частичная занятость'], dtype=object)

Удалим те записи, в которых нет ни salary_from, ни salary_to, т.к. такие данные для нас не представляют никакого интереса

In [108]:
df.dropna(subset=['salary_from', 'salary_to'], how='all', inplace=True)

In [109]:
df[['salary_from', 'salary_to']].isna().sum()

salary_from    6025
salary_to      5191
dtype: int64

In [110]:
df[df['salary_from'].isna() & df['salary_to'].isna()]

,vacancy_id,salary_from,salary_to,experience,city,employment_type


Теперь попробуем восстановить пропущенные данные по зарплатам с помощью алгоритма KNN

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error as mae

In [116]:
scaler = StandardScaler()